In [8]:
from __future__ import print_function, division
import numpy as np
import math
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import os
import csv
from scipy.optimize import minimize_scalar

# --------------------------------------------------
# OUTPUT DIRECTORY SETUP
# --------------------------------------------------
OUTPUT_DIR = "/home/hp/raytrace_work/raytrace_results/Electron_Density_Model_Analysis"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
print("Output directory: {}".format(OUTPUT_DIR))

# --------------------------------------------------
# Params class
# --------------------------------------------------
class Params(object):
    pass

prm = Params()
prm.r_chromo = 1.02
prm.r_corona = 1.05

# --------------------------------------------------
# ELECTRON DENSITY MODEL
# --------------------------------------------------
def density_only(prm, x, y, z):
    r2 = x*x + y*y + z*z
    r  = math.sqrt(r2)

    global r_corm16, r_corm6, r_corm2d5

    R0 = 6.955e5
    g1 = 3.09e8
    g2 = 1.58e8
    g3 = 2.51e6

    r_corm16  = prm.r_corona**(-16)
    r_corm6   = prm.r_corona**(-6)
    r_corm2d5 = prm.r_corona**(-2.5)

    A = np.zeros((4, 4))
    A[0] = [1., prm.r_chromo, prm.r_chromo**2, prm.r_chromo**3]
    A[1] = [1., prm.r_corona, prm.r_corona**2, prm.r_corona**3]
    A[2] = [0., 1., 2.*prm.r_chromo, 3.*prm.r_chromo**2]
    A[3] = [0., 1., 2.*prm.r_corona, 3.*prm.r_corona**2]
    A = np.linalg.inv(A)

    rhsa    = np.zeros(4)
    rhsa[0] = 5.7e11 * math.exp(-7.7e-4*(R0*(prm.r_chromo - 1) - 500))
    a       = np.dot(A, rhsa)

    if r < prm.r_chromo:
        Ne = 5.7e11 * math.exp(-7.7e-4*(R0*(r - 1) - 500))
    else:
        rm2d5     = r**(-2.5)
        rm6       = r**(-6)
        rm16      = r**(-16)
        cosTh     = abs(z / r)
        sqrtCosTh = math.sqrt(cosTh)
        t1 = 1.0 - 0.5*cosTh
        t2 = 1.0 - 0.95*cosTh
        t3 = 1.0 - sqrtCosTh

        if r < prm.r_corona:
            saito_rcor = g1*r_corm16*t1 + g2*r_corm6*t2 + g3*r_corm2d5*t3

            rhsb    = np.zeros(4)
            rhsb[1] = 1.
            rhsb[2] = (-438.9e6 * R0 *
                       math.exp(-7.7e-4*(R0*(prm.r_chromo - 1) - 500))
                       / saito_rcor)
            rhsb[3] = ((-16.*g1*(prm.r_corona**(-17))*t1
                        - 6.*g2*(prm.r_corona**(-7))*t2
                        - 2.5*g3*(prm.r_corona**(-3.5))*t3)
                       / saito_rcor)
            b = np.dot(A, rhsb)

            Ne = ((a[0] + r*(a[1] + r*(a[2] + r*a[3])))
                  + (b[0] + r*(b[1] + r*(b[2] + r*b[3]))) * saito_rcor)
        else:
            Ne = g1*rm16*t1 + g2*rm6*t2 + g3*rm2d5*t3

    return Ne

# --------------------------------------------------
# PLOT 1: Electron Density vs Radius
# --------------------------------------------------
r_vals  = np.linspace(1.0, 3.0, 4000)
Ne_vals = []

for r in r_vals:
    Ne = density_only(prm, r, 0.0, 0.0)
    Ne_vals.append(Ne)

Ne_vals = np.array(Ne_vals)

plt.figure()
plt.plot(r_vals, Ne_vals, linestyle='-', marker='o', markersize=4)
plt.xlabel('r ($R_\odot$)')
plt.ylabel('Electron Density Ne (cm^-3)')
plt.title('Electron Density vs Radius')
plt.xlim(1.0, 1.5)
plt.yscale('log')
plt.grid(True)
plt.tight_layout()
plot1_path = os.path.join(OUTPUT_DIR, "plot1_Ne_vs_radius.png")
plt.savefig(plot1_path, dpi=150)
print("Saved: {}".format(plot1_path))

print('Upper Boundary of Chromosphere = {}'.format(prm.r_chromo))
print('Lower Boundary of Corona       = {}'.format(prm.r_corona))

# --------------------------------------------------
# FIND MINIMUM IN TRANSITION REGION
# --------------------------------------------------
def middle_density(r, prm):
    return density_only(prm, r, 0.0, 0.0)

def find_minimum_middle(prm):
    result = minimize_scalar(
        middle_density,
        bounds=(prm.r_chromo, prm.r_corona),
        args=(prm,),
        method='bounded'
    )
    return result.x, result.fun

r_min, Ne_min = find_minimum_middle(prm)
print("Minimum occurs at r = {}".format(r_min))
print("Minimum Ne          = {}".format(Ne_min))

# --------------------------------------------------
# TRANSITION REGION COEFFICIENTS
# --------------------------------------------------
def transition_coefficients(prm):
    R0 = 6.955e5
    g1 = 3.09e8
    g2 = 1.58e8
    g3 = 2.51e6

    A = np.zeros((4, 4))
    A[0] = [1., prm.r_chromo, prm.r_chromo**2, prm.r_chromo**3]
    A[1] = [1., prm.r_corona, prm.r_corona**2, prm.r_corona**3]
    A[2] = [0., 1., 2.*prm.r_chromo, 3.*prm.r_chromo**2]
    A[3] = [0., 1., 2.*prm.r_corona, 3.*prm.r_corona**2]
    Ainv = np.linalg.inv(A)

    rhsa    = np.zeros(4)
    rhsa[0] = 5.7e11 * math.exp(-7.7e-4*(R0*(prm.r_chromo - 1) - 500))
    a       = np.dot(Ainv, rhsa)

    rcor       = prm.r_corona
    saito_rcor = g1*rcor**(-16) + g2*rcor**(-6) + g3*rcor**(-2.5)

    rhsb    = np.zeros(4)
    rhsb[1] = 1.
    rhsb[2] = (-438.9e6 * R0 *
               math.exp(-7.7e-4*(R0*(prm.r_chromo - 1) - 500))
               / saito_rcor)
    rhsb[3] = ((-16.*g1*(rcor**(-17))
                - 6.*g2*(rcor**(-7))
                - 2.5*g3*(rcor**(-3.5)))
               / saito_rcor)
    b = np.dot(Ainv, rhsb)

    return a, b

def Ne_explicit(r, prm, a, b):
    g1 = 3.09e8
    g2 = 1.58e8
    g3 = 2.51e6
    P  = a[0] + a[1]*r + a[2]*r**2 + a[3]*r**3
    Q  = b[0] + b[1]*r + b[2]*r**2 + b[3]*r**3
    S  = g1*r**(-16) + g2*r**(-6) + g3*r**(-2.5)
    return P + Q*S

def dNe_dr(r, prm, a, b):
    g1 = 3.09e8
    g2 = 1.58e8
    g3 = 2.51e6
    dP = a[1] + 2*a[2]*r + 3*a[3]*r**2
    Q  = b[0] + b[1]*r + b[2]*r**2 + b[3]*r**3
    dQ = b[1] + 2*b[2]*r + 3*b[3]*r**2
    S  = g1*r**(-16) + g2*r**(-6) + g3*r**(-2.5)
    dS = (-16*g1*r**(-17) - 6*g2*r**(-7) - 2.5*g3*r**(-3.5))
    return dP + dQ*S + Q*dS

# --------------------------------------------------
# COMPUTE TRANSITION REGION DATA
# --------------------------------------------------
a, b     = transition_coefficients(prm)
r_trans  = np.linspace(prm.r_chromo, prm.r_corona, 500)
Ne_original = [middle_density(r, prm) for r in r_trans]
Ne_poly     = [Ne_explicit(r, prm, a, b) for r in r_trans]
dNe_vals    = [dNe_dr(r, prm, a, b) for r in r_trans]

# --------------------------------------------------
# PLOT 2: Verification — Original vs Explicit
# --------------------------------------------------
plt.figure()
plt.plot(r_trans, Ne_original, label="Original density_only")
plt.plot(r_trans, Ne_poly, '--', label="Explicit polynomial form")
plt.xlabel("r (Solar Radii)")
plt.ylabel("Ne")
plt.legend()
plt.title("Verification: Original vs Explicit Expression")
plt.tight_layout()
plot2_path = os.path.join(OUTPUT_DIR, "plot2_verification_original_vs_explicit.png")
plt.savefig(plot2_path, dpi=150)
print("Saved: {}".format(plot2_path))

# --------------------------------------------------
# PLOT 3: Analytical Derivative dNe/dr
# --------------------------------------------------
plt.figure()
plt.plot(r_trans, dNe_vals)
plt.axhline(0, linestyle='--')
plt.xlabel("r (Solar Radii)")
plt.ylabel("dNe/dr")
plt.title("Analytical Derivative in Transition Region")
plt.tight_layout()
plot3_path = os.path.join(OUTPUT_DIR, "plot3_dNe_dr_transition.png")
plt.savefig(plot3_path, dpi=150)
print("Saved: {}".format(plot3_path))

Output directory: /home/hp/raytrace_work/raytrace_results/Electron_Density_Model_Analysis
Saved: /home/hp/raytrace_work/raytrace_results/Electron_Density_Model_Analysis/plot1_Ne_vs_radius.png
Upper Boundary of Chromosphere = 1.02
Lower Boundary of Corona       = 1.05
Minimum occurs at r = 1.02357774806
Minimum Ne          = 1524002.67608
Saved: /home/hp/raytrace_work/raytrace_results/Electron_Density_Model_Analysis/plot2_verification_original_vs_explicit.png
Saved: /home/hp/raytrace_work/raytrace_results/Electron_Density_Model_Analysis/plot3_dNe_dr_transition.png


In [9]:
# --------------------------------------------------
# HELPER: compute Ne at given r and theta
#
# theta = colatitude = angle from Z axis (pole)
# theta = 0  --> pole    (z=r,  cosTh = z/r = 1)
# theta = 90 --> equator (z=0,  cosTh = z/r = 0)
#
# cosTh = z/r = cos(theta)
# x = r * sin(theta)
# y = 0
# z = r * cos(theta)
# --------------------------------------------------
def Ne_at_r_theta(r, theta_deg):
    theta_rad = math.radians(theta_deg)
    x = r * math.sin(theta_rad)   # equatorial component
    y = 0.0
    z = r * math.cos(theta_rad)   # polar component
    return density_only(prm, x, y, z)

# --------------------------------------------------
# SPECIFIC VALUES OF R (Solar Radii)
# --------------------------------------------------
R_values = [1.1, 1.5, 2.0, 3.0]
R_labels = [u'R = 1.1 Rs',
            u'R = 1.5 Rs',
            u'R = 2.0 Rs',
            u'R = 3.0 Rs']
R_colors = [u'red', u'blue', u'green', u'purple']

# theta: 0 (pole) to 90 (equator)
theta_arr = np.linspace(0.0, 90.0, 500)

# --------------------------------------------------
# COMPUTE Ne vs theta for each R
# --------------------------------------------------
Ne_matrix = []

for R in R_values:
    Ne_row = []
    for th in theta_arr:
        Ne = Ne_at_r_theta(R, th)
        Ne_row.append(Ne)
    Ne_matrix.append(Ne_row)

Ne_matrix = np.array(Ne_matrix)   # shape: (4, 500)

# Print sample values
print("{:>8}  {:>12}  {:>12}  {:>12}  {:>12}".format(
    "theta", "R=1.1Rs", "R=1.5Rs",
    "R=2.0Rs", "R=3.0Rs"))
print("-" * 60)
for i in range(0, len(theta_arr), 50):
    print("{:>8.1f}  {:>12.4e}  {:>12.4e}  "
          "{:>12.4e}  {:>12.4e}".format(
              theta_arr[i],
              Ne_matrix[0, i],
              Ne_matrix[1, i],
              Ne_matrix[2, i],
              Ne_matrix[3, i]))
    

# --------------------------------------------------
# PLOT 4: Electron Density vs thetha (log scale)
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

for k in range(len(R_values)):
    ax.plot(theta_arr, Ne_matrix[k, :],
            linestyle=u'-',
            marker=u'o',
            markersize=2,
            linewidth=1.8,
            color=R_colors[k],
            label=R_labels[k])

ax.set_xlabel(
    u'Theta (degrees)\n'
    u'cosTh = z/r = cos(theta)  '
    u'[0=pole, 90=equator]')
ax.set_ylabel(u'Electron Density Ne (cm^-3)')
ax.set_title(
    u'Electron Density vs Theta\n'
    u'at 4 Radial Distances (log scale)')
ax.set_yscale(u'log')
ax.set_xlim(0, 90)
ax.set_xticks([0, 15, 30, 45, 60, 75, 90])
ax.legend(fontsize=10)
ax.grid(True, linestyle=u'--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(
    OUTPUT_DIR,
    "Ne_vs_theta_4R_logscale.png"), dpi=150)
plt.close()
print("Saved log scale plot.")

# --------------------------------------------------
# PLOT 5: Electron Density vs thetha (linear scale)
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

for k in range(len(R_values)):
    ax.plot(theta_arr, Ne_matrix[k, :],
            linestyle=u'-',
            marker=u'o',
            markersize=2,
            linewidth=1.8,
            color=R_colors[k],
            label=R_labels[k])

ax.set_xlabel(
    u'Theta (degrees)\n'
    u'cosTh = z/r = cos(theta)  '
    u'[0=pole, 90=equator]')
ax.set_ylabel(u'Electron Density Ne (cm^-3)')
ax.set_title(
    u'Electron Density vs Theta\n'
    u'at 4 Radial Distances (linear scale)')
ax.set_xlim(0, 90)
ax.set_xticks([0, 15, 30, 45, 60, 75, 90])
ax.legend(fontsize=10)
ax.grid(True, linestyle=u'--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(
    OUTPUT_DIR,
    "Ne_vs_theta_4R_linear.png"), dpi=150)
plt.close()
print("Saved linear scale plot.")

   theta       R=1.1Rs       R=1.5Rs       R=2.0Rs       R=3.0Rs
------------------------------------------------------------
     0.0    3.8083e+07    9.2877e+05    1.2579e+05    1.0840e+04
     9.0    3.9558e+07    1.1002e+06    1.5757e+05    1.4384e+04
    18.0    4.3948e+07    1.6105e+06    2.5219e+05    2.4964e+04
    27.1    5.1145e+07    2.4477e+06    4.0766e+05    4.2435e+04
    36.1    6.0973e+07    3.5923e+06    6.2070e+05    6.6573e+04
    45.1    7.3195e+07    5.0177e+06    8.8692e+05    9.7102e+04
    54.1    8.7514e+07    6.6916e+06    1.2011e+06    1.3376e+05
    63.1    1.0358e+08    8.5770e+06    1.5576e+06    1.7643e+05
    72.1    1.2103e+08    1.0635e+07    1.9516e+06    2.2546e+05
    81.2    1.3945e+08    1.2835e+07    2.3826e+06    2.8301e+05
Saved log scale plot.
Saved linear scale plot.
